In [1]:
import os
import sys
import warnings
warnings.filterwarnings('ignore')
# add the project root to the path so the src package can be imported
sys.path.append(os.path.abspath('..'))

import yaml
from ultralytics import YOLO
from src.utils import apply_smart_aug
from src.model_utils import smart_predict,export_enum_by_quad_using_model,analyze_quadrant_predictions,compare_best_vs_last
from src.model_callbacks import on_fit_epoch_end # It will ask you now immediatly (What Stage you are in right now?) Just Type your currently stage number
from src.generate_report import generate_report
import pandas as pd

%matplotlib inline

Patience Limit Is 20 Loaded Successfuly from the config file!


In [ ]:
with open('../configs/stage1.yaml','r') as f:

    stage1_config = yaml.safe_load(f)


with open('../configs/stage2.yaml','r') as f:

    stage2_config = yaml.safe_load(f)


with open('../configs/stage3.yaml','r') as f:

    stage3_config = yaml.safe_load(f)


with open(stage1_config['model_args']['data'],'r') as f:

    quad_data_yaml = yaml.safe_load(f)


with open(stage2_config['model_args']['data'],'r') as f:

    enum_data_yaml = yaml.safe_load(f)


with open(stage3_config['model_args']['data'],'r') as f:

    dis_data_yaml = yaml.safe_load(f)

In [3]:
stage1_config

{'paths': {'original_images_path': '../Data/Raw/DENTEX CHALLENGE 2023/Training_data/quadrant/xrays',
  'original_json_path': '../Data/Raw/DENTEX CHALLENGE 2023/Training_data/quadrant/train_quadrant.json',
  's1_main_path': '..\\Data\\Processed\\Stage 1 (Quadrant Detection)',
  'runs_s1_output': '..\\Runs\\Stage 1'},
 'model_args': {'data': '..\\Data\\Processed\\Stage 1 (Quadrant Detection)\\data.yaml',
  'model': '../Models/yolo26s.pt',
  'epochs': 50,
  'save': True,
  'imgsz': 1024,
  'batch': 8,
  'patience': 10,
  'optimizer': 'AdamW',
  'lr0': 0.001,
  'lrf': 0.01,
  'plots': True,
  'verbose': True,
  'device': 'cuda',
  'workers': 4,
  'project': 'Runs',
  'name': 'Stage 1',
  'save_dir': '..\\Runs\\Stage 1',
  'auto_augment': 'None',
  'augment': True,
  'mosaic': 0.0,
  'mixup': 0.0,
  'copy_paste': 0.0,
  'cutmix': 0.0,
  'hsv_h': 0.015,
  'hsv_s': 0.4,
  'hsv_v': 0.4,
  'degrees': 5.0,
  'translate': 0.05,
  'scale': 0.1,
  'shear': 0.0,
  'perspective': 0.0,
  'flipud': 0.0

In [4]:
stage2_config

{'paths': {'original_images_path': '../Data/Raw/DENTEX CHALLENGE 2023/Training_data/quadrant-enumeration/xrays',
  'original_json_path': '../Data/Raw/DENTEX CHALLENGE 2023/Training_data/quadrant-enumeration/train_quadrant_enumeration.json',
  's2_main_path': '..\\Data\\Processed\\Stage 2 (Enumeration Detection)',
  'runs_s2_output': '..\\Runs\\Stage 2',
  'diagnosis_quadrants_train_path': '..\\Data\\Processed\\Stage 2 (Enumeration Detection)\\Diagnosis Quadrants Train',
  'runs_s2_continued_output': '..\\Runs\\Stage 2 Continued'},
 'model_args': {'data': '..\\Data\\Processed\\Stage 2 (Enumeration Detection)\\data.yaml',
  'model': '../Models/yolo26s.pt',
  'epochs': 100,
  'save': True,
  'imgsz': 1024,
  'batch': 8,
  'patience': 20,
  'optimizer': 'AdamW',
  'lr0': 0.005,
  'lrf': 0.01,
  'plots': True,
  'verbose': True,
  'device': 'cuda',
  'workers': 4,
  'project': 'Runs',
  'name': 'Stage 2',
  'save_dir': '..\\Runs\\Stage 2',
  'auto_augment': 'None',
  'augment': False,
  'mo

In [5]:
stage3_config

{'paths': {'original_images_path': '../Data/Raw/DENTEX CHALLENGE 2023/Training_data/quadrant-enumeration-disease/xrays',
  'original_json_path': '../Data/Raw/DENTEX CHALLENGE 2023/Training_data/quadrant-enumeration-disease/train_quadrant_enumeration_disease.json',
  's3_main_path': '..\\Data\\Processed\\Stage 3 (Disease Classifier)',
  'runs_s3_output': '..\\Runs\\Stage 3'},
 'model_args': {'data': '..\\Data\\Processed\\Stage 3 (Disease Classifier)\\data.yaml'}}

In [6]:
quad_data_yaml

{'train': 'C:\\Users\\ibrah.HIMA\\OneDrive\\Desktop\\Full AI\\Projects\\GitHub - Kaggle Projects\\dental-xray-ai\\Data\\Processed\\Stage 1 (Quadrant Detection)\\train\\images',
 'val': 'C:\\Users\\ibrah.HIMA\\OneDrive\\Desktop\\Full AI\\Projects\\GitHub - Kaggle Projects\\dental-xray-ai\\Data\\Processed\\Stage 1 (Quadrant Detection)\\valid\\images',
 'test': 'C:\\Users\\ibrah.HIMA\\OneDrive\\Desktop\\Full AI\\Projects\\GitHub - Kaggle Projects\\dental-xray-ai\\Data\\Processed\\Stage 1 (Quadrant Detection)\\test\\images',
 'nc': 4,
 'names': ['Upper Right', 'Upper Left', 'Lower Left', 'Lower Right']}

In [7]:
enum_data_yaml

{'train': 'C:\\Users\\ibrah.HIMA\\OneDrive\\Desktop\\Full AI\\Projects\\GitHub - Kaggle Projects\\dental-xray-ai\\Data\\Processed\\Stage 2 (Enumeration Detection)\\train\\images',
 'val': 'C:\\Users\\ibrah.HIMA\\OneDrive\\Desktop\\Full AI\\Projects\\GitHub - Kaggle Projects\\dental-xray-ai\\Data\\Processed\\Stage 2 (Enumeration Detection)\\valid\\images',
 'test': 'C:\\Users\\ibrah.HIMA\\OneDrive\\Desktop\\Full AI\\Projects\\GitHub - Kaggle Projects\\dental-xray-ai\\Data\\Processed\\Stage 2 (Enumeration Detection)\\test\\images',
 'nc': 8,
 'names': [0, 1, 2, 3, 4, 5, 6, 7]}

In [8]:
dis_data_yaml 

{'train': 'C:\\Users\\ibrah.HIMA\\OneDrive\\Desktop\\Full AI\\Projects\\GitHub - Kaggle Projects\\dental-xray-ai\\Data\\Processed\\Stage 3 (Disease Classifier)\\train\\',
 'val': 'C:\\Users\\ibrah.HIMA\\OneDrive\\Desktop\\Full AI\\Projects\\GitHub - Kaggle Projects\\dental-xray-ai\\Data\\Processed\\Stage 3 (Disease Classifier)\\valid\\',
 'test': 'C:\\Users\\ibrah.HIMA\\OneDrive\\Desktop\\Full AI\\Projects\\GitHub - Kaggle Projects\\dental-xray-ai\\Data\\Processed\\Stage 3 (Disease Classifier)\\test\\',
 'nc': 5,
 'names': ['Impacted', 'Caries', 'Periapical', 'Deep Caries', 'No Disease']}